# 07 — Cross-Dataset Validation

This notebook evaluates the generalization capability of the trained machine learning models using independent external datasets and compares performance across different clinical distributions.

In [8]:
import os
import sys

sys.path.append(os.path.abspath("../src"))

In [9]:


import warnings
warnings.filterwarnings("ignore")

import joblib
import pandas as pd

from sklearn.model_selection import train_test_split

from evaluation import (
    evaluate_model,
    evaluate_models
)



In [10]:

X_train = pd.read_csv("../data/processed/X_train_scaled.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()

X_uci_train = pd.read_csv("../data/processed/X_uci_train.csv")
y_uci_train = pd.read_csv("../data/processed/y_uci_train.csv").squeeze()

X_uci_cleveland = pd.read_csv("../data/processed/X_uci_cleveland.csv")
y_uci_cleveland = pd.read_csv("../data/processed/y_uci_cleveland.csv").squeeze()



In [11]:


models = {

    "Random Forest":
    joblib.load("../saved_models/random_forest.pkl"),

    "Decision Tree":
    joblib.load("../saved_models/decision_tree.pkl"),

    "SVM":
    joblib.load("../saved_models/svm.pkl"),

    "Logistic Regression":
    joblib.load("../saved_models/logistic_regression.pkl"),

    "KNN":
    joblib.load("../saved_models/knn.pkl"),

    "XGBoost":
    joblib.load("../saved_models/xgboost.pkl")
}



In [12]:

for model in models.values():

    model.fit(
        X_train,
        y_train
    )



In [13]:

uci_results = evaluate_models(

    models,

    X_uci_train,

    y_uci_train
)

uci_results.sort_values(
    by="Accuracy",
    ascending=False
)



,Accuracy,Precision,Recall,F1 Score,AUC,Hamming Loss,Jaccard Score,Log Loss,MCC
SVM,0.592532,0.658667,0.667568,0.663087,0.614085,0.407468,0.495984,1.023615,0.147767
Decision Tree,0.576299,0.681063,0.554054,0.611028,0.604609,0.423701,0.439914,11.234810,0.160498
Logistic Regression,0.574675,0.663636,0.591892,0.625714,0.612272,0.425325,0.455301,1.162784,0.138146
XGBoost,0.558442,0.697581,0.467568,0.559871,0.656543,0.441558,0.388764,1.014028,0.162472
Random Forest,0.508117,0.731034,0.286486,0.411650,0.658328,0.491883,0.259169,0.948781,0.147711
KNN,0.496753,0.758621,0.237838,0.362140,0.546979,0.503247,0.221106,6.971481,0.155358


In [14]:

combined_X = pd.concat(

    [X_train, X_uci_train],

    ignore_index=True
)

combined_y = pd.concat(

    [y_train, y_uci_train],

    ignore_index=True
)



In [15]:

X_train_combined, X_test_combined, y_train_combined, y_test_combined = train_test_split(

    combined_X,

    combined_y,

    test_size=0.2,

    stratify=combined_y,

    random_state=42
)


In [16]:

for model in models.values():

    model.fit(

        X_train_combined,

        y_train_combined
    )


In [17]:

combined_results = evaluate_models(

    models,

    X_test_combined,

    y_test_combined
)

combined_results.sort_values(
    by="Accuracy",
    ascending=False
)


,Accuracy,Precision,Recall,F1 Score,AUC,Hamming Loss,Jaccard Score,Log Loss,MCC
XGBoost,0.883803,0.894118,0.910180,0.902077,0.956293,0.116197,0.821622,0.272214,0.759430
Random Forest,0.880282,0.893491,0.904192,0.898810,0.954041,0.119718,0.816216,0.264814,0.752350
KNN,0.859155,0.884848,0.874251,0.879518,0.934183,0.140845,0.784946,0.768039,0.710116
Logistic Regression,0.852113,0.874251,0.874251,0.874251,0.929474,0.147887,0.776596,0.339219,0.694764
SVM,0.848592,0.864706,0.880240,0.872404,0.923435,0.151408,0.773684,0.347324,0.686457
Decision Tree,0.841549,0.886076,0.838323,0.861538,0.862915,0.158451,0.756757,4.043711,0.678104


In [18]:


cleveland_results = evaluate_models(

    models,

    X_uci_cleveland,

    y_uci_cleveland
)

cleveland_results.sort_values(

    by="Accuracy",

    ascending=False
)



,Accuracy,Precision,Recall,F1 Score,AUC,Hamming Loss,Jaccard Score,Log Loss,MCC
SVM,0.786184,0.728395,0.848921,0.784053,0.862830,0.213816,0.644809,0.481131,0.581380
Logistic Regression,0.779605,0.746575,0.784173,0.764912,0.864138,0.220395,0.619318,0.468824,0.558314
KNN,0.756579,0.694611,0.834532,0.758170,0.829540,0.243421,0.610526,1.289964,0.526084
XGBoost,0.743421,0.682635,0.820144,0.745098,0.810377,0.256579,0.593750,0.607362,0.499542
Random Forest,0.740132,0.672414,0.841727,0.747604,0.793264,0.259868,0.596939,0.692636,0.499716
Decision Tree,0.628289,0.591549,0.604317,0.597865,0.658709,0.371711,0.426396,6.737623,0.252422


In [19]:


comparison = pd.concat(

    {

        "Train Mendeley → Test UCI":
        uci_results,

        "Train Combined → Test Combined":
        combined_results,

        "Train Combined → Test Cleveland":
        cleveland_results

    },

    axis=1
)

comparison



Train Mendeley → Test UCI                                \
                                     Accuracy Precision    Recall  F1 Score   
Random Forest                        0.508117  0.731034  0.286486  0.411650   
Decision Tree                        0.576299  0.681063  0.554054  0.611028   
SVM                                  0.592532  0.658667  0.667568  0.663087   
Logistic Regression                  0.574675  0.663636  0.591892  0.625714   
KNN                                  0.496753  0.758621  0.237838  0.362140   
XGBoost                              0.558442  0.697581  0.467568  0.559871   

                                                                               \
                          AUC Hamming Loss Jaccard Score   Log Loss       MCC   
Random Forest        0.658328     0.491883      0.259169   0.948781  0.147711   
Decision Tree        0.604609     0.423701      0.439914  11.234810  0.160498   
SVM                  0.614085     0.407468      0.495984   1.023615  0.147767   
Logistic Regression  0.612272     0.425325      0.455301   1.162784  0.138146   
KNN                  0.546979     0.503247      0.221106   6.971481  0.155358   
XGBoost              0.656543     0.441558      0.388764   1.014028  0.162472   

                    Train Combined → Test Combined  ...            \
                                          Accuracy  ...       MCC   
Random Forest                             0.880282  ...  0.752350   
Decision Tree                             0.841549  ...  0.678104   
SVM                                       0.848592  ...  0.686457   
Logistic Regression                       0.852113  ...  0.694764   
KNN                                       0.859155  ...  0.710116   
XGBoost                                   0.883803  ...  0.759430   

                    Train Combined → Test Cleveland                      \
                                           Accuracy Precision    Recall   
Random Forest                              0.740132  0.672414  0.841727   
Decision Tree                              0.628289  0.591549  0.604317   
SVM                                        0.786184  0.728395  0.848921   
Logistic Regression                        0.779605  0.746575  0.784173   
KNN                                        0.756579  0.694611  0.834532   
XGBoost                                    0.743421  0.682635  0.820144   

                                                                              \
                     F1 Score       AUC Hamming Loss Jaccard Score  Log Loss   
Random Forest        0.747604  0.793264     0.259868      0.596939  0.692636   
Decision Tree        0.597865  0.658709     0.371711      0.426396  6.737623   
SVM                  0.784053  0.862830     0.213816      0.644809  0.481131   
Logistic Regression  0.764912  0.864138     0.220395      0.619318  0.468824   
KNN                  0.758170  0.829540     0.243421      0.610526  1.289964   
XGBoost              0.745098  0.810377     0.256579      0.593750  0.607362   

                               
                          MCC  
Random Forest        0.499716  
Decision Tree        0.252422  
SVM                  0.581380  
Logistic Regression  0.558314  
KNN                  0.526084  
XGBoost              0.499542  

[6 rows x 27 columns]

In [20]:

cardiacx = joblib.load(
    "../saved_models/cardiacx.pkl"
)


In [21]:


cardiacx.fit(

    X_train_combined,

    y_train_combined
)


[15:41:47] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

[15:41:47] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

[15:41:47] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

[15:41:47] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

[15:41:47] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.



In [22]:


cardiacx_results = evaluate_model(

    cardiacx,

    X_uci_cleveland,

    y_uci_cleveland
)

pd.DataFrame(

    cardiacx_results,

    index=["CARDIACX"]
)



,Accuracy,Precision,Recall,F1 Score,AUC,Hamming Loss,Jaccard Score,Log Loss,MCC
CARDIACX,0.756579,0.699387,0.820144,0.754967,0.83693,0.243421,0.606383,0.529035,0.522628


In [23]:


final_models = models.copy()

final_models["CARDIACX"] = cardiacx

final_results = evaluate_models(

    final_models,

    X_uci_cleveland,

    y_uci_cleveland
)

final_results.sort_values(

    by="Accuracy",

    ascending=False
)



,Accuracy,Precision,Recall,F1 Score,AUC,Hamming Loss,Jaccard Score,Log Loss,MCC
SVM,0.786184,0.728395,0.848921,0.784053,0.862830,0.213816,0.644809,0.481131,0.581380
Logistic Regression,0.779605,0.746575,0.784173,0.764912,0.864138,0.220395,0.619318,0.468824,0.558314
KNN,0.756579,0.694611,0.834532,0.758170,0.829540,0.243421,0.610526,1.289964,0.526084
CARDIACX,0.756579,0.699387,0.820144,0.754967,0.836930,0.243421,0.606383,0.529035,0.522628
XGBoost,0.743421,0.682635,0.820144,0.745098,0.810377,0.256579,0.593750,0.607362,0.499542
Random Forest,0.740132,0.672414,0.841727,0.747604,0.793264,0.259868,0.596939,0.692636,0.499716
Decision Tree,0.628289,0.591549,0.604317,0.597865,0.658709,0.371711,0.426396,6.737623,0.252422


In [24]:
uci_results.to_csv(
    "../results/uci_results.csv",
    index=False
)

combined_results.to_csv(
    "../results/combined_results.csv",
    index=False
)

cleveland_results.to_csv(
    "../results/cleveland_results.csv",
    index=False
)

In [2]:
import matplotlib.pyplot as plt

def save_dataframe_as_image(
    df,
    filename,
    title=None,
    figsize=(10, 3),
    fontsize=11,
    dpi=300
):
    """
    Save a pandas DataFrame as a high-quality PNG image.

    Parameters
    ----------
    df : pandas.DataFrame
    filename : str
        Example: "../docs/images/mendeley_results.png"
    title : str, optional
    figsize : tuple
    fontsize : int
    dpi : int
    """

    fig, ax = plt.subplots(figsize=figsize)
    ax.axis("off")

    if title:
        plt.title(title, fontsize=14, fontweight="bold", pad=12)

    table = ax.table(
        cellText=df.values,
        colLabels=df.columns,
        cellLoc="center",
        loc="center"
    )

    table.auto_set_font_size(False)
    table.set_fontsize(fontsize)
    table.scale(1.2, 1.6)

    # Style header
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight="bold")
            cell.set_height(0.12)
        else:
            cell.set_height(0.10)

    plt.savefig(
        filename,
        dpi=dpi,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.close(fig)

    print(f"✅ Saved: {filename}")

In [4]:

save_dataframe_as_image(
    final_results,
    "../docs/images/uci_results.png",
    title="Train on Mendeley → Test on UCI"
)

save_dataframe_as_image(
    combined_results,
    "../docs/images/combined_results.png",
    title="Combined Dataset Evaluation"
)

save_dataframe_as_image(
    cleveland_results,
    "../docs/images/cleveland_results.png",
    title="Cleveland External Evaluation"
)

NameError: name 'final_results' is not defined